# SNEMI3D, CREMI, MICrONS and CellMap in 3D with Isaac Sim

This notebook draws four EM volumes as 3D scenes in Isaac Sim: one surface mesh per neurite (per organelle for CellMap), the synaptic clefts where the dataset has them, and three image sections on the far faces of the block.

It runs on the machine that has Isaac Sim (here the lab's DGX Spark), with the kernel **Isaac Sim (connectome)**. That kernel is the Python of Isaac Sim; `bash sim/install_kernel.sh` registers it once. Four steps per dataset:

1. **Mesh** the labels of a crop into a scene package (`sim/mesh_crop.py`).
2. **Start** Isaac Sim. Isaac Lab's `AppLauncher` is used if Isaac Lab is installed, otherwise Isaac Sim's own `SimulationApp`.
3. **Build** the USD stage (`sim/scene.py`) and **render** it with Replicator.
4. **Show** each scene in an interactive 3D view in the notebook (`sim/web_view.py`), beside the Isaac Sim pictures.

`SimulationApp` is started with the reduced Kit app `sim/connectome.viewer.kit` (RTX renderer and Replicator only), because the full Isaac Sim app returns empty pictures on a build whose Isaac Sim extensions do not load (the state of the Spark build on 23 November 2026). If the application cannot start in this kernel, or it returns empty pictures, step 3 falls back to the Kit runtime of the same Isaac Sim build in a separate process (`sim/render_kit.py`), which gives the same `scene.usd` and pictures.

The crops are made by `render/data/fetch_*.py`. Paths come from environment variables, with the defaults of the Spark.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

HERE = Path.cwd()
SIM = HERE / "sim"                                   # scene.py, mesh_crop.py, render_kit.py
assert (SIM / "scene.py").exists(), "start this notebook in research/connectome"
ISAACSIM_PATH = Path(os.environ.get("ISAACSIM_PATH", Path.home() / "IsaacSim/_build/linux-aarch64/release"))
CROPS = Path(os.environ.get("CONNECTOME_CROPS", Path.home() / "connectome/data/crops"))
OUT = Path(os.environ.get("CONNECTOME_OUT", Path.home() / "connectome/out/isaac"))
# A Python with numpy, scipy, scikit-image and pillow, for the meshing step and the interactive views
# (Isaac Sim's Python has none of them): the conda environment "isaac".
PYTHON = os.environ.get("CONNECTOME_PYTHON", str(Path.home() / "miniconda3/envs/isaac/bin/python"))
# The kernel of Isaac Sim puts its own Python 3.12 on PYTHONPATH; that Python must not see it.
PYTHON_ENV = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME")}
DATASETS = {"snemi3d": "SNEMI3D", "cremi3d": "CREMI3D", "microns": "MICrONS minnie65",
            "cellmap": "CellMap crop 313 (mouse liver)"}
TOP = {"snemi3d": 60, "cremi3d": 60, "microns": 60, "cellmap": 0}   # largest objects to mesh; 0 = all
VIEWS = 2         # pictures around each block
SIZE = 1024       # width and height of a picture
WEB_TRIANGLES = 250_000   # triangle budget of each interactive view
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SIM))
for name in DATASETS:
    print(name, json.loads((CROPS / name / "meta.json").read_text())["shape"], "voxels (z, y, x)")

## 1. Meshes

Marching cubes on each label, after taking every n-th voxel in the plane so that the meshing voxel is near isotropic. Lengths become micrometres. Only the largest objects are meshed, to keep the scene readable; for CellMap every organelle is meshed and coloured by its class, and the cytosol between them is left out. The step runs in a separate Python (`CONNECTOME_PYTHON`), because the Python of Isaac Sim has no scipy or scikit-image.

In [ ]:
manifests = {}
for name in DATASETS:
    done = subprocess.run([PYTHON, str(SIM / "mesh_crop.py"), str(CROPS / name), str(OUT / name), "--top", str(TOP[name])],
                          capture_output=True, text=True, env=PYTHON_ENV)
    assert done.returncode == 0, done.stderr[-2000:]
    m = manifests[name] = json.loads((OUT / name / "manifest.json").read_text())
    print(f"{DATASETS[name]}: {len(m['objects'])} meshes, {sum(o['triangles'] for o in m['objects']):,} triangles, "
          f"block {m['size'][0]:.2f} x {m['size'][1]:.2f} x {m['size'][2]:.2f} um, "
          f"meshing voxel {[round(v * 1000) for v in m['mesh_voxel_um']]} nm (z, y, x)")

## 2. Isaac Sim / Isaac Lab

`AppLauncher` (Isaac Lab) and `SimulationApp` (Isaac Sim) both start the Omniverse application inside this Python process, which is why the notebook needs the Isaac Sim kernel. Isaac Lab is tried first; without it, Isaac Sim alone is enough for drawing.

In [ ]:
%%capture startup
# The start-up log of Isaac Sim is long; it is kept in `startup` (startup.show() prints it).
app, how = None, None
try:
    from isaaclab.app import AppLauncher            # Isaac Lab, if installed
    app = AppLauncher(headless=True, enable_cameras=True).app
    how = "Isaac Lab AppLauncher"
except Exception as lab_error:
    try:
        from isaacsim import SimulationApp          # Isaac Sim alone
        # The reduced Kit app of sim/ (RTX renderer and Replicator only). The full Isaac Sim app
        # returns empty pictures on a build whose Isaac Sim extensions do not load.
        app = SimulationApp({"headless": True, "width": SIZE, "height": SIZE},
                            experience=str(SIM / "connectome.viewer.kit"))
        how = "Isaac Sim SimulationApp"
    except Exception as sim_error:
        print("Isaac Lab:", type(lab_error).__name__, lab_error)
        print("Isaac Sim:", type(sim_error).__name__, sim_error)

In [ ]:
from IPython.display import Markdown, display


def say(text):
    """Show a line under the cell. Isaac Sim takes over print() once it has started."""
    display(Markdown(text))


say(f"Started in this kernel: **{how or 'nothing; the Kit runtime will be used in a separate process'}**")

## 3. Stage and pictures

`scene.build` fills a USD stage from a package: `/World/Volume/Objects/<name>` for the meshes, `/World/Volume/Slices` for the image sections, a wire frame of the block, lights, a ground plane and a camera. The stage is saved as `scene.usd` in the package folder, and can be opened in the Isaac Sim window.

In [ ]:
def render_in_app(package):
    """Build and render in the application of this kernel. Returns the picture files."""
    import omni.usd
    import omni.replicator.core as rep
    import numpy as np
    from PIL import Image
    import scene
    omni.usd.get_context().new_stage()
    stage = omni.usd.get_context().get_stage()
    center, radius = scene.build(stage, package)
    usd = str(package / "scene.usd")
    stage.GetRootLayer().Export(usd)
    omni.usd.get_context().open_stage(usd)
    stage = omni.usd.get_context().get_stage()
    product = rep.create.render_product(scene.CAMERA, (SIZE, SIZE))
    rgb = rep.AnnotatorRegistry.get_annotator("rgb")
    rgb.attach([product])
    files = []
    for n in range(VIEWS):
        scene.set_view(stage, center, radius, -60.0 + 360.0 * n / VIEWS)
        for _ in range(100):
            app.update()
        rep.orchestrator.step()
        image = np.asarray(rgb.get_data())
        if image.size == 0 or image[..., :3].std() < 1:
            raise RuntimeError("the renderer returned an empty picture")
        files.append(package / f"render_{n}.png")
        Image.fromarray(image[..., :3]).save(files[-1])
    rgb.detach(); product.destroy()
    return files


def render_with_kit(package):
    """Build and render with the Kit runtime of Isaac Sim, in a separate process."""
    cmd = [str(ISAACSIM_PATH / "kit/kit"), str(SIM / "connectome.viewer.kit"), "--no-window",
           "--ext-folder", "extscache", "--ext-folder", "exts",
           "--exec", f"{SIM / 'render_kit.py'} {package} --views {VIEWS} --width {SIZE} --height {SIZE}"]
    log = subprocess.run(cmd, cwd=ISAACSIM_PATH, capture_output=True, text=True)
    files = [package / f"render_{n}.png" for n in range(VIEWS)]
    if log.returncode or not all(f.exists() for f in files):
        raise RuntimeError(log.stdout[-2000:] + log.stderr[-2000:])
    return files


pictures, route = {}, {}
for name in DATASETS:
    package = OUT / name
    for f in package.glob("render_*.png"):
        f.unlink()
    try:
        if app is None:
            raise RuntimeError("no application in this kernel")
        pictures[name], route[name] = render_in_app(package), how
    except Exception as e:
        say(f"{name}: in-kernel render failed ({type(e).__name__}: {str(e)[:200]}); using the Kit runtime")
        pictures[name], route[name] = render_with_kit(package), "Kit runtime of Isaac Sim"
    say(f"{name}: rendered by **{route[name]}**, {len(pictures[name])} pictures")

## 4. The four volumes

Each volume is shown twice: an interactive view, then the pictures of Isaac Sim. The interactive view is made by `sim/web_view.py` from the same package as the USD stage (same meshes, colours, image sections and block), simplified to about `WEB_TRIANGLES` triangles, and drawn with WebGL in the notebook; it starts from the camera of the first Isaac Sim picture. Drag to turn, shift-drag or right-drag to move, the wheel to zoom; click an object to show it alone, double-click to reset. The slider keeps only the largest objects; the legend names the classes that have their own colour. Each view is also written as `viewer.html` in the package folder, which a browser opens as it is.

In [ ]:
from IPython.display import HTML, Image as Picture

for name, title in DATASETS.items():
    m = manifests[name]
    say(f"**{title}**: {m['size'][0]:.2f} x {m['size'][1]:.2f} x {m['size'][2]:.2f} um, voxel "
          f"{m['voxel_nm']} nm (z, y, x), {len(m['objects'])} meshes, rendered by {route[name]}")
    done = subprocess.run([PYTHON, str(SIM / "web_view.py"), str(OUT / name), "--title", title,
                           "--triangles", str(WEB_TRIANGLES)], capture_output=True, text=True, env=PYTHON_ENV)
    assert done.returncode == 0, done.stderr[-2000:]
    display(HTML((OUT / name / "viewer.html").read_text()))
    for f in pictures[name]:
        display(Picture(filename=str(f), width=480))

## What the pictures show

- One colour per object: the 60 largest of each neurite crop. CellMap: one colour per organelle class (the colours of the CellMap model viewer), all 131 organelles of the crop. Yellow: synaptic clefts (CREMI: manual labels; MICrONS: the automatic cleft segmentation). SNEMI3D has no cleft labels.
- The labels of MICrONS are the automatic segmentation; those of SNEMI3D, CREMI and CellMap are manual.
- Surfaces are made from the label grid. Sections of SNEMI3D, CREMI and MICrONS are 29 to 40 nm thick, so the steps between sections on the surfaces come from the acquisition, not from the tissue. CellMap is FIB-SEM with 8 nm voxels in all three directions, so its surfaces have no such steps.
- The interactive views are simplified (vertex clustering to a cell of a few tens of nanometres), so small objects lose detail there; the Isaac Sim pictures use the full meshes.
- Each `scene.usd` under the output folder opens in the Isaac Sim window (`isaac-sim.sh`, or `isaac-sim.streaming.sh` to view it from another machine).

In [ ]:
# Isaac Sim stops when the kernel is shut down or restarted. app.close() is not called here:
# it ends the kernel process at once, and a command-line run (jupyter-execute) then loses its outputs.
say("Isaac Sim is still running in this kernel; restart or shut down the kernel to stop it." if app is not None
    else "No application was started in this kernel.")
